<a href="https://colab.research.google.com/github/leesungyeon-dot/10thML/blob/week_4/4%EC%A3%BC%EC%B0%A8_%EC%98%88%EC%8A%B5%EA%B3%BC%EC%A0%9C_%EC%9D%B4%EC%84%B1%EC%97%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. 데이터 로드 및 전처리

먼저 `sklearn.datasets`에서 위스콘신 유방암 데이터셋을 로드하고, 훈련 세트와 테스트 세트로 나눕니다.

In [1]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# 데이터 로드
cancer = load_breast_cancer()
X = cancer.data
y = cancer.target

# 훈련 세트와 테스트 세트 분리
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"훈련 세트 크기: {X_train.shape}")
print(f"테스트 세트 크기: {X_test.shape}")

훈련 세트 크기: (455, 30)
테스트 세트 크기: (114, 30)


### 2. XGBoost (Python 래퍼) 모델

`xgboost` 라이브러리의 DMatrix를 사용하여 모델을 훈련하고 평가합니다.

In [2]:
import xgboost as xgb

# DMatrix 생성 (XGBoost Python 래퍼 전용)
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)

# 모델 파라미터 설정
params = {
    'objective': 'binary:logistic', # 이진 분류를 위한 로지스틱 회귀
    'eval_metric': 'logloss', # 평가 지표
    'eta': 0.1, # 학습률
    'max_depth': 3, # 트리의 최대 깊이
    'seed': 42
}

# 모델 훈련
num_rounds = 100
model_xgb_native = xgb.train(params, dtrain, num_rounds)

# 예측
y_pred_xgb_native = model_xgb_native.predict(dtest)
y_pred_xgb_native_binary = (y_pred_xgb_native > 0.5).astype(int)

# 평가
print("--- XGBoost (Python 래퍼) 평가 ---")
print(f"정확도: {accuracy_score(y_test, y_pred_xgb_native_binary):.4f}")
print("분류 보고서:")
print(classification_report(y_test, y_pred_xgb_native_binary))

--- XGBoost (Python 래퍼) 평가 ---
정확도: 0.9561
분류 보고서:
              precision    recall  f1-score   support

           0       0.95      0.93      0.94        43
           1       0.96      0.97      0.97        71

    accuracy                           0.96       114
   macro avg       0.96      0.95      0.95       114
weighted avg       0.96      0.96      0.96       114



### 3. XGBoost (사이킷런 래퍼) 모델

`XGBClassifier`를 사용하여 사이킷런 스타일로 모델을 훈련하고 평가합니다.

In [3]:
from xgboost import XGBClassifier

# 모델 초기화 및 훈련
model_xgb_sklearn = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    eta=0.1,
    max_depth=3,
    random_state=42,
    use_label_encoder=False # Deprecation warning 방지
)
model_xgb_sklearn.fit(X_train, y_train)

# 예측
y_pred_xgb_sklearn = model_xgb_sklearn.predict(X_test)

# 평가
print("--- XGBoost (사이킷런 래퍼) 평가 ---")
print(f"정확도: {accuracy_score(y_test, y_pred_xgb_sklearn):.4f}")
print("분류 보고서:")
print(classification_report(y_test, y_pred_xgb_sklearn))

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [07:57:27] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


--- XGBoost (사이킷런 래퍼) 평가 ---
정확도: 0.9561
분류 보고서:
              precision    recall  f1-score   support

           0       0.95      0.93      0.94        43
           1       0.96      0.97      0.97        71

    accuracy                           0.96       114
   macro avg       0.96      0.95      0.95       114
weighted avg       0.96      0.96      0.96       114



### 4. LightGBM 모델

`lightgbm` 라이브러리의 `LGBMClassifier`를 사용하여 모델을 훈련하고 평가합니다.

## 1. 캐글 산탄데르 고객 만족 예측 (XGBoost)

### 1.1 데이터 로드 및 전처리

먼저 캐글 API를 사용하여 `santander-customer-satisfaction` 데이터셋을 다운로드하고 로드!

In [5]:
# 필요한 라이브러리 설치 (처음 실행 시 필요)
!pip install kaggle

import os

# Colab Secrets에서 Kaggle API 자격 증명 로드 (선택 사항)
# 만약 kaggle.json 파일이 이미 업로드되어 있다면 이 부분은 건너뛸 수 있습니다.
# from google.colab import userdata
# os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
# os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

# 캐글 데이터셋 다운로드 및 압축 해제
!kaggle competitions download -c santander-customer-satisfaction
!unzip -o santander-customer-satisfaction.zip -d santander-customer-satisfaction

You must authenticate before you can call the Kaggle API.
Follow the instructions to authenticate at: https://github.com/Kaggle/kaggle-cli/blob/main/docs/README.md#authentication
unzip:  cannot find or open santander-customer-satisfaction.zip, santander-customer-satisfaction.zip.zip or santander-customer-satisfaction.zip.ZIP.


In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

# 데이터 로드
# 다운로드된 파일 경로에 맞게 수정해주세요.
df = pd.read_csv('santander-customer-satisfaction/train.csv')

# 'ID' 컬럼은 예측에 불필요하므로 제거합니다.
df = df.drop('ID', axis=1)

# 특징(X)과 타겟(y) 분리
X = df.drop('TARGET', axis=1)
y = df['TARGET']

# 훈련 세트와 테스트 세트 분리 (클래스 불균형을 고려하여 stratify 적용)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"훈련 세트 크기: {X_train.shape}")
print(f"테스트 세트 크기: {X_test.shape}")
print(f"훈련 세트 타겟 분포:\n{y_train.value_counts(normalize=True)}")
print(f"테스트 세트 타겟 분포:\n{y_test.value_counts(normalize=True)}")

FileNotFoundError: [Errno 2] No such file or directory: 'santander-customer-satisfaction/train.csv'

### 1.2 XGBoost 베이스라인 모델 학습 및 평가

먼저 기본 설정으로 XGBoost 모델을 학습하고 성능을 평가합니다.

In [7]:
import xgboost as xgb

# XGBClassifier 모델 초기화
# 고객 만족도 예측은 이진 분류 문제이므로 objective='binary:logistic'을 사용합니다.
# 평가 지표는 ROC AUC가 중요하므로 eval_metric='auc'를 사용합니다.
model_xgb_baseline = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='auc',
    random_state=42,
    use_label_encoder=False # 최신 버전에서는 DeprecationWarning 방지를 위해 False로 설정
)

# 모델 학습
model_xgb_baseline.fit(X_train, y_train)

# 예측
y_pred_baseline = model_xgb_baseline.predict(X_test)
y_pred_proba_baseline = model_xgb_baseline.predict_proba(X_test)[:, 1] # ROC AUC를 위한 확률 예측

# 평가
print("--- XGBoost 베이스라인 모델 평가 ---")
print(f"정확도: {accuracy_score(y_test, y_pred_baseline):.4f}")
print(f"ROC AUC: {roc_auc_score(y_test, y_pred_proba_baseline):.4f}")
print("분류 보고서:")
print(classification_report(y_test, y_pred_baseline))

--- XGBoost 베이스라인 모델 평가 ---
정확도: 0.9561
ROC AUC: 0.9908
분류 보고서:
              precision    recall  f1-score   support

           0       0.95      0.93      0.94        43
           1       0.96      0.97      0.97        71

    accuracy                           0.96       114
   macro avg       0.96      0.95      0.95       114
weighted avg       0.96      0.96      0.96       114



/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [07:59:26] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


### 1.3 XGBoost 하이퍼파라미터 튜닝 (GridSearchCV)

모델의 성능을 최적화하기 위해 `GridSearchCV`를 사용하여 하이퍼파라미터를 튜닝합니다. 주요 하이퍼파라미터는 `n_estimators`, `learning_rate`, `max_depth` 등입니다. (시간이 오래 걸릴 수 있으므로, 처음에는 적은 범위로 테스트하고 점차 늘려가는 것을 권장합니다.)

In [ ]:
from sklearn.model_selection import GridSearchCV

# 튜닝할 하이퍼파라미터 범위 설정
param_grid = {
    'n_estimators': [100, 200, 300],  # 트리의 개수
    'learning_rate': [0.01, 0.05, 0.1], # 학습률
    'max_depth': [3, 5, 7],           # 트리의 최대 깊이
    'colsample_bytree': [0.7, 0.8, 0.9], # 각 트리마다 feature 샘플링 비율
    'subsample': [0.7, 0.8, 0.9]      # 각 트리마다 데이터 샘플링 비율
}

# GridSearchCV 객체 생성
# scoring='roc_auc'는 캐글 대회 평가 지표에 맞추기 위함입니다.
grid_search = GridSearchCV(
    estimator=xgb.XGBClassifier(
        objective='binary:logistic',
        eval_metric='auc',
        random_state=42,
        use_label_encoder=False
    ),
    param_grid=param_grid,
    scoring='roc_auc',
    cv=3, # 3-겹 교차 검증
    n_jobs=-1, # 가능한 모든 코어 사용
    verbose=2
)

# GridSearchCV 실행
grid_search.fit(X_train, y_train)

# 최적의 하이퍼파라미터 및 최고 점수 출력
print("--- 하이퍼파라미터 튜닝 결과 ---")
print(f"최적의 하이퍼파라미터: {grid_search.best_params_}")
print(f"최고 ROC AUC 점수 (훈련 세트): {grid_search.best_score_:.4f}")

Fitting 3 folds for each of 243 candidates, totalling 729 fits


### 1.4 최적화된 XGBoost 모델 평가

최적의 하이퍼파라미터로 다시 모델을 학습하고 테스트 세트에서 성능을 평가합니다.

In [ ]:
# 최적의 파라미터로 모델 생성
model_xgb_tuned = grid_search.best_estimator_

# 예측
y_pred_tuned = model_xgb_tuned.predict(X_test)
y_pred_proba_tuned = model_xgb_tuned.predict_proba(X_test)[:, 1]

# 평가
print("--- 최적화된 XGBoost 모델 평가 ---")
print(f"정확도: {accuracy_score(y_test, y_pred_tuned):.4f}")
print(f"ROC AUC: {roc_auc_score(y_test, y_pred_proba_tuned):.4f}")
print("분류 보고서:")
print(classification_report(y_test, y_pred_tuned))

In [4]:
import lightgbm as lgb

# 모델 초기화 및 훈련
model_lgbm = lgb.LGBMClassifier(
    objective='binary',
    metric='binary_logloss',
    n_estimators=100,
    learning_rate=0.1,
    num_leaves=31,
    random_state=42
)
model_lgbm.fit(X_train, y_train)

# 예측
y_pred_lgbm = model_lgbm.predict(X_test)

# 평가
print("--- LightGBM 평가 ---")
print(f"정확도: {accuracy_score(y_test, y_pred_lgbm):.4f}")
print("분류 보고서:")
print(classification_report(y_test, y_pred_lgbm))

[LightGBM] [Info] Number of positive: 286, number of negative: 169
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000394 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4548
[LightGBM] [Info] Number of data points in the train set: 455, number of used features: 30
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.628571 -> initscore=0.526093
[LightGBM] [Info] Start training from score 0.526093
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best 

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
